# 08 — A Flow controls the Crew
The Crew researches. The Flow validates input, tracks state, limits retries,
and routes either to a report or an explicit evidence gap.



**Basic lesson 8 of 13.**


## Problem

Crew collaboration alone does not define request validation or failure routing.


## Concept

A typed Flow owns state, branches, and a bounded retry around the Crew.

The Flow is deterministic orchestration around probabilistic work. Typed state carries the request and result; a router chooses the next path based on checked values, not on a free-form model promise.


## Where this fits in the whole workshop

```text
USER · CLI · API
   │
   ▼
 13 PROJECT        run_assistant(): memory in → Flow → report out
   │
   ├─ 06 MEMORY          recall this customer's saved constraint
   ├─ 09 CONTROL         validate the question before any model call
   ├─ 08 FLOW            typed state · route · one bounded retry   ◀ YOU ARE HERE
   │    └─ 07 CREW            Researcher → Analyst → Critic
   │         └─ 02 AGENT LOOP      model decides → tool runs → observation → answer
   │              ├─ 01 MODEL          the LLM call itself
   │              ├─ 03 TOOLS          typed Python functions the agent may call
   │              ├─ 04 MCP            tools served by a separate process
   │              └─ 05 KNOWLEDGE      cited passages from company documents
   └─ 09 CONTROL         validate the report → human approval → mock email

 around everything:
   10 RESILIENCE      checkpoint and resume
   11 OBSERVABILITY   trace what actually ran
   12 EVALUATION      checks that fail loudly
```

Every lesson adds one row to this system. The rows below the marker are still the same code from earlier lessons.


## Architecture of this lesson

```text
request ──▶ validate ──▶ Research Crew ──▶ report check
                               ▲                    │
                               └── retry, at most 1 ┘
                                             │
                           valid ──▶ report / approval route
                         invalid ──▶ evidence-gap route
```

validate → Crew → evidence check → report or insufficient-evidence route. A tool result or validation failure is an observation, not an instruction the model may bypass.


## Step 1 — Instantiate typed Flow state and supply the request fields

Instantiate typed Flow state and supply the request fields.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[2] if "__file__" in globals() else Path.cwd()))

from workshop.decision_flow import DecisionFlow

flow = DecisionFlow()
output = flow.kickoff(inputs={
    "question": "Should Northstar AI pilot Meridian AI?",
    "customer_id": "customer-a",
    "customer_context": "Customer A requires private VPC deployment.",
})



## Inspect the path, not only the answer
retry_count is capped. An evidence failure is a valid controlled result.



## Step 2 — Inspect report, retry count, and failure route after kickoff

Inspect report, retry count, and failure route after kickoff.


In [ ]:
print("Flow output:", output)
print("Retry count:", flow.state.retry_count)
print("Failure:", flow.state.failure or "none")
print("Report:", flow.state.report)



## Bonus scenario — lead scoring
A Flow needs no model to route work (inspired by CrewAI's *Lead Score* flow).
Typed state, a router, and one listener per branch: the same shape as above.



In [ ]:
from crewai.flow.flow import Flow, listen, router, start
from pydantic import BaseModel


class LeadState(BaseModel):
    budget_usd: int = 0
    team_size: int = 0
    score: int = 0


class LeadFlow(Flow[LeadState]):
    @start()
    def score_lead(self):
        self.state.score = min(self.state.budget_usd // 1000, 60) + min(self.state.team_size, 40)

    @router(score_lead)
    def route(self):
        return "hot" if self.state.score >= 70 else "nurture"

    @listen("hot")
    def call_today(self):
        return f"Score {self.state.score}: call today"

    @listen("nurture")
    def send_newsletter(self):
        return f"Score {self.state.score}: add to newsletter"


print(LeadFlow().kickoff(inputs={"budget_usd": 50_000, "team_size": 30}))
print(LeadFlow().kickoff(inputs={"budget_usd": 5_000, "team_size": 4}))


## Failure mode to know about

A weak report should not be retried forever. The Flow stops after two attempts and reports an evidence gap; a failed run is a valid controlled outcome.


## Production implication

Keep routing and retry counts in code, and make failure an explicit outcome. Persist only the state needed to resume safely.


## Your turn

Inspect retry_count and predict the branch for a report with one source.

**Checkpoint:** The run ends with a report or an explicit gap after at most two attempts.
